# Component-level margin attribution & causal tests

Turns the baseline/contextual decomposition into a component-level story: which components write the
frequency-linked **baseline** and which write answer/alternative-specific **contextual** support.

The final (normed) residual is a sum of component contributions: embedding + each layer's attention
block + each layer's MLP block. With the final RMSNorm scale held at its full-residual value, the
decomposition through the norm is **exact** (the norm is linear in the residual for a fixed scale), so
each component's contribution to the answer-vs-alternative margin is
$$\Delta z_{i,k}=\langle \tilde x_{i,k},\,W_U[a_i]-W_U[b_i]\rangle,\quad \sum_k \Delta z_{i,k}=M_i.$$
Splitting $\tilde x_{i,k}=\bar x_k+\Delta\tilde x_{i,k}$ (leave-one-out mean) gives baseline vs
contextual contributions per component.

- **C1 Attribution**: per-component total / baseline / contextual margin; which components push toward
  the alternative, which carry the baseline, which carry answer-specific context.
- **C2 Causal ablation**: mean-ablate the top alternative-favoring components; margin shift vs random.
- **C3 Paired-paraphrase patching** (cleanest causal): patch success components into the failed
  paraphrase of the SAME fact; measure margin/recovery. Controls fact, answer, relation, frequency.
- **C4 Baseline source**: which components build $\bar h$ and its frequency correlation.

Needs `rw_core.py`, `mech_core.py`, `mech_runner.py`, `component_core.py`. Run `test_component_core.py`.

## 0. Config + data

In [ ]:
import numpy as np, json, gc, torch, re
import rw_core as rw, mech_core as mc, mech_runner as mr, component_core as cc
from datasets import load_dataset
from collections import defaultdict
import pandas as pd

DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DTYPE=torch.float16 if DEVICE=="cuda" else torch.float32
QA="Answer with a short factual answer.\nQuestion: {q}\nAnswer:"
TEMPLATES=[QA,"Q: {q}\nA:","Please answer concisely.\n{q}\nAnswer:","{q} The answer is"]
N_ITEMS=300; CAPS=dict(attr=150, abl=150, pair=80, src=200)

MODELS=[
 "meta-llama/Llama-3.1-8B","meta-llama/Llama-3.2-3B","meta-llama/Llama-3.2-1B",
 "meta-llama/Llama-3.2-3B-Instruct","Qwen/Qwen2.5-3B","Qwen/Qwen2.5-3B-Instruct",
 "Qwen/Qwen2.5-7B","mistralai/Mistral-7B-v0.1",
]
# component capture is heavier (hooks on every block); start with MODELS[:2] to time it.

ds=load_dataset("akariasai/PopQA",split="test")
def aliases(r):
    a=r["possible_answers"]
    if isinstance(a,str):
        try: a=json.loads(a)
        except: a=[a]
    return a
ITEMS=[{"q":str(r["question"]),"gold":aliases(r),"rel":str(r.get("prop","na"))} for r in ds]
REL=defaultdict(list)
for it in ITEMS:
    for a in it["gold"]: REL[it["rel"]].append(a)
np.random.default_rng(0).shuffle(ITEMS); ITEMS=ITEMS[:N_ITEMS]
print("items per model:",len(ITEMS))

## 1. Run C1/C2/C3/C4 on all models (per-experiment isolation)

In [ ]:
COMP={}
for name in MODELS:
    print("="*70); print(name,flush=True)
    try:
        ctx=mr.make_ctx(name,DEVICE,DTYPE,ITEMS,REL,QA,TEMPLATES)
        out={}
        for key,fnc in [
            ("C1_attr", lambda: mr.exp_component_attribution(ctx, max_items=CAPS["attr"])),
            ("C2_abl",  lambda: mr.exp_component_ablation(ctx, max_items=CAPS["abl"], top_k=5)),
            ("C3_pair", lambda: mr.exp_paired_component_patch(ctx, max_facts=CAPS["pair"])),
            ("C4_src",  lambda: mr.exp_baseline_source(ctx, ctx["freq"], max_items=CAPS["src"])),
        ]:
            try: out[key]=fnc(); print(f"  {key} done",flush=True)
            except Exception as e:
                import traceback; traceback.print_exc(); out[key]={"status":"error","error":f"{type(e).__name__}: {e}"}
                print(f"  {key} FAILED: {type(e).__name__}: {str(e)[:140]}",flush=True)
        COMP[name]=out
    except Exception as e:
        import traceback; traceback.print_exc(); COMP[name]={"status":"error","error":f"{type(e).__name__}: {e}"}
    finally:
        try: mr.free_ctx(ctx); del ctx
        except Exception: pass
        torch.cuda.empty_cache(); gc.collect()
json.dump(COMP,open("component_attribution.json","w"),indent=2,default=float)
print("\nsucceeded:",len([k for k,v in COMP.items() if "status" not in v]),"/",len(COMP))

## 2. C1 — component attribution (which components write baseline vs contextual)
Top components pushing toward the selected alternative (most negative answer-minus-alternative margin),
split by total / baseline / contextual; and the top components writing answer-specific context.

In [ ]:
for nm,v in COMP.items():
    if "status" in v or "status" in v.get("C1_attr",{}): continue
    a=v["C1_attr"]
    print("###",nm.split("/")[-1],f"(n={a['n']}; sum base={a['sum_baseline']:.2f}, ctx={a['sum_contextual']:.2f}, total={a['sum_total']:.2f})")
    print("   top alternative-favoring (total):   ", ", ".join(f"{c}({s:+.2f})" for c,s in a["top_alternative_total"][:5]))
    print("   ... of which BASELINE-driven:        ", ", ".join(f"{c}({s:+.2f})" for c,s in a["top_alternative_baseline"][:5]))
    print("   ... of which CONTEXTUAL-driven:      ", ", ".join(f"{c}({s:+.2f})" for c,s in a["top_alternative_contextual"][:5]))
    print("   top ANSWER-supporting (contextual):  ", ", ".join(f"{c}({s:+.2f})" for c,s in a["top_answer_contextual"][:5]))

## 3. C2 — causal ablation of top margin components vs random
Mean-ablating the top alternative-favoring components should shift the margin toward gold MORE than
random components. A reliable margin shift is mechanistically meaningful even if full recovery is low.

In [ ]:
rows=[]
for nm,v in COMP.items():
    if "status" in v or "status" in v.get("C2_abl",{}): continue
    a=v["C2_abl"]
    rows.append({"model":nm.split("/")[-1],"n":a["n"],
                 "top_shift":round(a["shift_top"]["mean_shift"],3),
                 "rand_shift":round(a["shift_random"]["mean_shift"],3),
                 "top_frac_improved":round(a["shift_top"]["frac_improved"],3),
                 "first_tok_recov":round(a["first_token_recovery"],3),
                 "sel_freq_shift":round(a["mean_selected_freq_shift"],3),
                 "top_comps":",".join(c.replace("attn.","a").replace("mlp.","m") for c in a["top_components"])})
print(pd.DataFrame(rows).to_string(index=False))
print("\ntop_shift >> rand_shift = the identified components causally carry the alternative's advantage.")

## 4. C3 — success->failure paired-paraphrase component patching (cleanest causal)
Patch component outputs from the SUCCESSFUL paraphrase into the FAILED one (same fact/answer/relation/
frequency). Positive mean_shift and recovery show a small set of late components carries the success/
failure difference.

In [ ]:
rows=[]
for nm,v in COMP.items():
    if "status" in v or "status" in v.get("C3_pair",{}): continue
    a=v["C3_pair"]
    if "status" in a: print(nm,"->",a.get("status")); continue
    row={"model":nm.split("/")[-1],"n_pairs":a.get("n_pairs")}
    for bn in ["late_attn","late_mlp","late_all"]:
        if bn in a:
            row[f"{bn}_shift"]=round(a[bn]["mean_shift"],3); row[f"{bn}_recov"]=round(a[bn]["recovery"],3)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))
print("\nlate_all should give the largest shift/recovery; if late_attn alone ~ late_all, attention")
print("carries the difference. This controls for fact/answer/relation/frequency (paired design).")

## 5. C4 — source of the readout baseline
Which components contribute most to the mean residual and to corr(baseline, frequency). Removing a
component's mean and seeing corr(base,freq) drop identifies the frequency-baseline writers.

In [ ]:
for nm,v in COMP.items():
    if "status" in v or "status" in v.get("C4_src",{}): continue
    a=v["C4_src"]
    print("###",nm.split("/")[-1],f"(n={a['n']}, corr(base,freq) full={a['corr_baseline_freq_full']:.3f})")
    print("   components whose mean most affects corr(base,freq):",
          ", ".join(f"{c}({d:+.3f})" for c,d in a["top_components_by_freqcorr_contribution"][:6]))
    print("   components with largest mean-residual norm:        ",
          ", ".join(f"{c}({nr:.2f})" for c,nr in a["top_components_by_meannorm"][:6]))

## Notes
- The attribution is **exact**: with the final RMSNorm scale fixed at its full-residual value, the
  norm is linear in the residual, so per-component margin contributions sum to the final margin. (This
  requires the captured block outputs to sum to the pre-norm residual, which holds in pre-norm
  transformers: residual = emb + sum(attn) + sum(mlp).)
- **C1** separates baseline-writing from context-writing components; expect the baseline (frequency-
  linked) to concentrate in different components than answer-specific context.
- **C2** is the causal check on attribution; report top_shift vs rand_shift (a reliable shift suffices;
  full recovery may be low because write strength is binding).
- **C3** is the cleanest causal experiment: paired paraphrases control fact/answer/relation/frequency,
  so a positive patch effect isolates the components carrying success vs failure.
- **C4** localizes the baseline source; pair with the alignment result (baseline vs frequency / BOS).
- Component capture is heavier (hooks on every block). Start with `MODELS[:2]`; raise `CAPS` for finals.